In [ ]:
from typing import Dict, Sequence

import numpy as np
import pandas as pd
from scipy import stats

ALPHA = 0.05
TABLE_METRICS = [
    "score",
    "accuracy_seen_avg",
    "ece_seen_avg",
    "backward_transfer",
    "sce_final",
]
FMT_STRS = {
    "_": ("{:02.2f}", "{:.2f}"),
    "sce_final": ("{:.3f}", "{:.3f}"),
    "ace_seen_avg": ("{:.3f}", "{:.3f}"),
}

LOWER_IS_BETTER = {
    "ece_seen_avg",
    "sce_final",
    "ace_seen_avg",
}


df = pd.read_csv("run.csv")

In [ ]:
df["score"] = 1 / 2 * (df["accuracy_seen_avg"] + (1 - df["ece_seen_avg"]))

In [ ]:
def significance_test(
    df: pd.DataFrame, metric: str, reference: str
) -> Dict[str, tuple[float, float]]:
    equal_var = False
    samples: Sequence[np.ndarray] = []
    methods: Sequence[str] = []

    for method, group in df.groupby("method"):
        assert isinstance(method, str)
        group_samples = np.abs(group[metric].to_numpy())
        if np.any(np.isnan(group_samples)):
            continue
        samples.append(group_samples)
        methods.append(method)

    # One way ANOVA
    f_oneway_result = stats.f_oneway(*samples, equal_var=equal_var)
    if f_oneway_result.pvalue >= ALPHA:
        print(
            f"WARNING: ANOVA test for metric {metric} is not significant (p={f_oneway_result.pvalue:.4f})"
        )

    reference_idx = methods.index(reference)
    result = stats.tukey_hsd(*samples, equal_var=equal_var)

    return {
        method: (result.pvalue[i, reference_idx], result.statistic[i, reference_idx])
        for i, method in enumerate(methods)
    }

In [ ]:
reference_method = "tball"


def to_display_table(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    p_values: Dict[str, Dict[str, tuple[float, float]]] = {}
    for m in TABLE_METRICS:
        p_values[m] = significance_test(df, m, reference=reference_method)

    df = df.groupby("method").aggregate({m: ["mean", "std"] for m in TABLE_METRICS})

    rows = []

    for method, data in df.iterrows():
        method: str
        row = {"method": method}
        for key in TABLE_METRICS:
            p_value, test_statistic = p_values[key].get(method, (1.0, 0.0))
            label = key
            col_max = df[key]["mean"].max()
            col_min = df[key]["mean"].min()

            value = data[(key, "mean")]
            mean = float(value * 100)
            std = data[(key, "std")] * 100

            # if std is NaN, replace with 0.0
            if pd.isna(std):
                std = 0.0

            # Determine if this value should be bolded
            if key not in LOWER_IS_BETTER:
                bold = value >= col_max
            else:
                bold = value <= col_min

            # If the method is in UNFAIR_COMPARISONS, never bold
            italics = False
            fmt_mean, fmt_std = FMT_STRS.get(key) or FMT_STRS["_"]
            label = rf"{{\small {label}}}"
            row[label] = rf"{fmt_mean.format(mean)} $\pm$ {fmt_std.format(std)}"
            if bold:
                row[label] = rf"\textbf{{{row[label]}}}"
            if italics:
                row[label] = rf"\textit{{{row[label]}}}"

            # Significance annotation
            sig_higher = r"$^{\uparrow}$"
            sig_lower = r"$^{\downarrow}$"
            if p_value < ALPHA:
                # The test-statistic points the wrong way when the mean is negative
                # Maybe an absolute value is taken somewhere?
                if mean < 0:
                    test_statistic = -test_statistic
                if test_statistic > 0:
                    row[label] += sig_higher
                else:
                    row[label] += sig_lower

            row[label] = r"\small " + row[label]

        rows.append(row)

    table = pd.DataFrame(rows)
    table = table.set_index("method")
    table.index.name = None
    return table

In [ ]:
for dataset in df["dataset"].unique():
    dataset_table = to_display_table(df[df["dataset"] == dataset])
    filename = f"table-{dataset}.tex"
    with open(filename, "w") as f:
        dataset_table.to_latex(
            f, column_format="l" + "l" * (len(dataset_table.columns))
        )
    print(f"Saved table for {dataset} to {filename}")